In [140]:
!pip install numpy pandas scikit-learn demoji textblob
!python -m textblob.download_corpora


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
[nltk_data] Downloading package brown to /home/sat/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/sat/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /home/sat/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/sat/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to /home/sat/nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to
[nltk_data]     /home/sat/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
Finished.


In [141]:
import numpy as np
import pandas as pd
import sklearn as skl
import demoji as dem
import textblob as tb
dem.download_codes()


/tmp/ipykernel_11825/3331246871.py:6: FutureWarning: The demoji.download_codes attribute is deprecated and will be removed from demoji in a future version. It is an unused attribute as emoji codes are now distributed directly with the demoji package.
  dem.download_codes()


# Reading Data & Data Preprocessing

In [142]:
df = pd.read_csv("semeval-tweets/twitter-training-data.txt", sep="\t", header=None, names=["id", "sentiment", "tweet"])
print(f"NaN tweets: {df['tweet'].isna().sum()}")
df = df.dropna(subset=["tweet"]).reset_index(drop=True)

for i in range(len(df["tweet"])):
  tweet = df.loc[i, "tweet"]
  tweetNoEmoji = dem.replace_with_desc(tweet)
  df.loc[i, "tweet"] = tweetNoEmoji
df

NaN tweets: 1


,id,sentiment,tweet
0,335104872099066692,positive,Felt privileged to play Foo Fighters songs on ...
1,796528524030124618,positive,@AaqibAfzaal Pakistan may be an Islamic countr...
2,760964834217238632,positive,Happy Birthday to the coolest golfer in Bali! ...
3,147713180324524046,negative,@SimpplyA TMILLS is going to Tucson! But the 2...
4,732302280474120023,negative,Hmmmmm where are the #BlackLivesMatter when ma...
...,...,...,...
45020,660374218263817235,neutral,Sunday Cinema | Paul McCartney &amp; David Gil...
45021,739323365061217061,neutral,"14-Aug The day of independence, The day of S..."
45022,681369726697754114,positive,"9 September has arrived, which means Apple's n..."
45023,922217029064536808,positive,So I'll see you all tomorrow for some fun fill...


In [143]:
# test set
dftest = pd.read_csv("semeval-tweets/twitter-test1.txt", sep="\t", header=None, names=["id", "sentiment", "tweet"])
print(f"NaN tweets: {df['tweet'].isna().sum()}")
dftest = dftest.dropna(subset=["tweet"]).reset_index(drop=True)

for i in range(len(dftest["tweet"])):
  tweet = dftest.loc[i, "tweet"]
  tweetNoEmoji = dem.replace_with_desc(tweet)
  dftest.loc[i, "tweet"] = tweetNoEmoji
dftest

NaN tweets: 0


,id,sentiment,tweet
0,163361196206957578,neutral,Candids: Heading to the Chateau Marmont in Wes...
1,768006053969268950,negative,@Dont__KAY_me omg same I was reading it in sch...
2,742616104384772304,neutral,Watching MTV Hits! The Wanted Chasing the sun!
3,102313285628711403,neutral,"Bing one-ups knowledge graph, hires Encyclopae..."
4,653274888624828198,neutral,"On Thursday, concealed-carry gun license holde..."
...,...,...,...
3526,897427103459983282,positive,"@heavyoilcountry right you are, they could end..."
3527,494418545323766863,neutral,We will have to think about continuing in UPA:...
3528,556020888361859268,positive,ladies come throw that ass and niggas come thr...
3529,683639126147064420,neutral,@MandiieeKillaaa are you going to laidback Luk...


In [144]:
print(df["sentiment"].value_counts(dropna=False))

sentiment
neutral     20748
positive    15960
negative     8317
Name: count, dtype: int64


### Preprocess

In [145]:
# lower case everything
df["tweet"] = df['tweet'].str.lower()
dftest["tweet"] = dftest['tweet'].str.lower()
df

,id,sentiment,tweet
0,335104872099066692,positive,felt privileged to play foo fighters songs on ...
1,796528524030124618,positive,@aaqibafzaal pakistan may be an islamic countr...
2,760964834217238632,positive,happy birthday to the coolest golfer in bali! ...
3,147713180324524046,negative,@simpplya tmills is going to tucson! but the 2...
4,732302280474120023,negative,hmmmmm where are the #blacklivesmatter when ma...
...,...,...,...
45020,660374218263817235,neutral,sunday cinema | paul mccartney &amp; david gil...
45021,739323365061217061,neutral,"14-aug the day of independence, the day of s..."
45022,681369726697754114,positive,"9 september has arrived, which means apple's n..."
45023,922217029064536808,positive,so i'll see you all tomorrow for some fun fill...


In [146]:
# Remove urls
df["tweet"] = df['tweet'].str.replace(r'https?://\S+', '', regex=True)
dftest["tweet"] = dftest['tweet'].str.replace(r'https?://\S+', '', regex=True)

In [147]:
# Removing whitespace and \n
df["tweet"] = df["tweet"].str.replace(r'\s+', ' ', regex=True).str.strip()
dftest["tweet"] = dftest["tweet"].str.replace(r'\s+', ' ', regex=True).str.strip()

In [148]:
# remove punctuation
#df["tweet"] = df['tweet'].str.replace(r'[^\w\s]', '', regex=True)
#dftest["tweet"] = dftest['tweet'].str.replace(r'[^\w\s]', '', regex=True)
#df["tweet"].dtype

In [149]:
# Remove non english characters
for i in range(len(df["tweet"])):
  tweet = df.loc[i, "tweet"]
  df.loc[i, "tweet"] = tweet.encode('ascii',errors='ignore').decode('ascii')

for i in range(len(dftest["tweet"])):
  tweet = dftest.loc[i, "tweet"]
  dftest.loc[i, "tweet"] = tweet.encode('ascii',errors='ignore').decode('ascii')

In [150]:
# Lemmatisation with Texblob
for i in range(len(df["tweet"])):
    blob = tb.TextBlob(df.loc[i, "tweet"])
    lemmatised = []
    for word, tag in blob.tags:
        if tag.startswith('VB'):
            lemmatised.append(word.lemmatize('v'))
        elif tag.startswith('NN'):
            lemmatised.append(word.lemmatize('n'))
        elif tag.startswith('JJ'):
            lemmatised.append(word.lemmatize('a'))
        else:
            lemmatised.append(word)
    df.loc[i, "tweet"] = " ".join(lemmatised)

df

,id,sentiment,tweet
0,335104872099066692,positive,felt privilege to play foo fighter song on gui...
1,796528524030124618,positive,@ aaqibafzaal pakistan may be an islamic count...
2,760964834217238632,positive,happy birthday to the cool golfer in bali @ tj...
3,147713180324524046,negative,@ simpplya tmills be go to tucson but the 29th...
4,732302280474120023,negative,hmmmmm where be the blacklivesmatter when matt...
...,...,...,...
45020,660374218263817235,neutral,sunday cinema | paul mccartney & amp david gil...
45021,739323365061217061,neutral,14-aug the day of independence the day of sacr...
45022,681369726697754114,positive,9 september have arrive which mean apple 's ne...
45023,922217029064536808,positive,so i 'll see you all tomorrow for some fun fil...


In [151]:
for i in range(len(dftest["tweet"])):
    blob = tb.TextBlob(dftest.loc[i, "tweet"])
    lemmatised = []
    for word, tag in blob.tags:
        if tag.startswith('VB'):
            lemmatised.append(word.lemmatize('v'))
        elif tag.startswith('NN'):
            lemmatised.append(word.lemmatize('n'))
        elif tag.startswith('JJ'):
            lemmatised.append(word.lemmatize('a'))
        else:
            lemmatised.append(word)
    dftest.loc[i, "tweet"] = " ".join(lemmatised)

dftest

,id,sentiment,tweet
0,163361196206957578,neutral,candids heading to the chateau marmont in west...
1,768006053969268950,negative,@ dont__kay_me omg same i be read it in school...
2,742616104384772304,neutral,watch mtv hit the wanted chase the sun
3,102313285628711403,neutral,bing one-ups knowledge graph hire encyclopaedi...
4,653274888624828198,neutral,on thursday concealed-carry gun license holder...
...,...,...,...
3526,897427103459983282,positive,@ heavyoilcountry right you be they could end ...
3527,494418545323766863,neutral,we will have to think about continue in upa ma...
3528,556020888361859268,positive,lady come throw that as and niggas come throw ...
3529,683639126147064420,neutral,@ mandiieekillaaa be you go to laidback luke t...


### N-Gram and then BoW the dataset

In [152]:
X, y = "tweet", "sentiment"

In [153]:
df[y] = df[y].replace({"positive": 2, "neutral": 1, "negative": 0}).astype(int)
dftest[y] = dftest[y].replace({"positive": 2, "neutral": 1, "negative": 0}).astype(int)

In [154]:
df[y].value_counts()

sentiment
1    20748
2    15960
0     8317
Name: count, dtype: int64

In [155]:
dftest[y].value_counts()

sentiment
1    1504
2    1470
0     557
Name: count, dtype: int64

# Classifier 1: SVM

In [ ]:
def modelFunc():
  bestC = 0
  bestF1 = -np.inf
  bestWeights = {}
  bestF1Weights = -np.inf
  for c in [0.5, 1.0, 1.3, 1.5, 2.0]:
    for weights in [{0: 0.7, 1: 0.15, 2: 0.3}, {0: 2.0, 1: 0.5, 2: 1.0}, {0: 1.5, 1: 0.3, 2: 0.5}]:
      
      pipeCV = skl.pipeline.Pipeline([
        ('tfidf', skl.feature_extraction.text.TfidfVectorizer(ngram_range=(1, 3), sublinear_tf=True, min_df=3, max_df=0.95)),
        ('svm', skl.svm.LinearSVC(C = c, max_iter = 10000, class_weight = weights))
      ])
      scores = skl.model_selection.cross_val_score(pipeCV, df[X], df[y], cv=5, scoring='f1_macro')
      print(f"--> weights = {weights}: CV F1 = {scores.mean():.4f} (+/- {scores.std():.4f})")
      bestWeights = weights if scores.mean() > bestF1Weights else bestWeights
      bestF1Weights = scores.mean() if scores.mean() > bestF1Weights else bestF1Weights
    print(f"C={c}: CV F1={scores.mean():.4f} (+/- {scores.std():.4f})")
    bestC = c if scores.mean() > bestF1 else bestC
    bestF1 = scores.mean() if scores.mean() > bestF1 else bestF1


  # now train on best C
  pipe = skl.pipeline.Pipeline([
    ('tfidf', skl.feature_extraction.text.TfidfVectorizer(ngram_range=(1, 3), sublinear_tf=True, min_df=3, max_df=0.95)),
    ('svm', skl.svm.LinearSVC(C=bestC, max_iter=10000, class_weight={
      0: 0.7, 
      1: 0.15, 
      2: 0.3
    }))
  ])

  return bestC, bestF1, pipe

In [176]:
bestC, bestF1, pipe = modelFunc()
print("DONE CV.")
bestC, bestF1

C=0.5: CV F1=0.6228 (+/- 0.0020)
C=1.0: CV F1=0.6221 (+/- 0.0028)
C=1.3: CV F1=0.6201 (+/- 0.0022)
C=1.5: CV F1=0.6192 (+/- 0.0025)
C=2.0: CV F1=0.6154 (+/- 0.0034)
DONE CV.


(0.5, np.float64(0.6228335618557991))

In [177]:
# Use the best model
pipe.fit(df[X], df[y])
preds = pipe.predict(dftest[X])
f1 = skl.metrics.f1_score(dftest[y], preds, labels=[0, 2], average='macro')
f1

0.6438446346280448

In [178]:
mcm = skl.metrics.multilabel_confusion_matrix(dftest[y], preds, labels=[0, 1, 2])

for i, label in enumerate(["negative", "neutral", "positive"]):
    tn, fp, fn, tp = mcm[i].ravel()
    print(f"{label}: TP={tp}, FP={fp}, FN={fn}, TN={tn}")

negative: TP=328, FP=263, FN=229, TN=2711
neutral: TP=937, FP=435, FN=567, TN=1592
positive: TP=1088, FP=480, FN=382, TN=1581


In [179]:
report = skl.metrics.classification_report(dftest[y], preds)
print(report)

              precision    recall  f1-score   support

           0       0.55      0.59      0.57       557
           1       0.68      0.62      0.65      1504
           2       0.69      0.74      0.72      1470

    accuracy                           0.67      3531
   macro avg       0.64      0.65      0.65      3531
weighted avg       0.67      0.67      0.67      3531

